# News Data Sources for Forecasting Questions

This notebook demonstrates two approaches to using news articles as data sources for generating forecasting questions:

1. **News Search** (`NewsSeedGenerator`) - Uses Google News to search for recent articles matching specific queries
2. **Top Aggregated News** (`GdeltSeedGenerator`) - Uses [GDELT](https://www.gdeltproject.org/) to access a massive database of global news articles

Both approaches follow the same pipeline pattern but offer different trade-offs in coverage, control, and use cases.

## Install the SDK

In [10]:
%pip install lightningrod-ai python-dotenv

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/sign-up?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [11]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Build the pipeline

Let's first define the configuration that will be shared across both pipelines to generate binary questions from news sources.

In [ ]:
from lightningrod import BinaryAnswerType, WebSearchLabeler, QuestionRenderer

answer_type = BinaryAnswerType()

labeler = WebSearchLabeler(
    answer_type=answer_type,
    confidence_threshold=0.5,
)

renderer = QuestionRenderer(
    answer_type=answer_type,
)

### Approach 1: News Search (Google News)

The `NewsSeedGenerator` searches Google News for articles matching your query. You can specify date ranges, search queries, and how many articles to fetch per interval.

In [13]:
from datetime import datetime
from lightningrod import NewsSeedGenerator, QuestionGenerator, QuestionPipeline

news_seed_generator = NewsSeedGenerator(
    start_date=datetime(2025, 9, 1),
    end_date=datetime(2025, 10, 1),
    interval_duration_days=7,  # Split date range into intervals of this many days
    articles_per_search=20,  # Maximum number of articles to fetch per search query per interval
    search_query="AI technology announcements",
)

question_generator = QuestionGenerator(
    instructions=(
        "Generate forward-looking questions about AI technology announcements. "
        "Questions should be about future events or outcomes that can be verified later."
    ),
    examples=[
        "Will OpenAI release a new model in Q2 2025?",
        "Will Google announce a new AI product this month?",
        "Will Apple integrate AI features into iOS 19?",
    ],
    bad_examples=[
        "What did OpenAI announce?",
        "Who is the CEO of Google?",
        "When was ChatGPT released?",
    ],
    answer_type=answer_type,
)


news_search_pipeline_config = QuestionPipeline(
    seed_generator=news_seed_generator,
    question_generator=question_generator,
    labeler=labeler,
    renderer=renderer,
)

### Approach 2: Top Aggregated News (GDELT)

The `GdeltSeedGenerator` fetches articles at intervals defined by `interval_duration_days` - it does not fetch articles for every day (unless you set `interval_duration_days=1`), but instead steps forward by the specified interval between each batch. This provides access to a massive database of global news articles.

In [14]:

from lightningrod import GdeltSeedGenerator, FilterCriteria

gdelt_seed_generator = GdeltSeedGenerator(
    start_date=datetime(2025, 10, 1),
    end_date=datetime(2025, 11, 1),
    interval_duration_days=7,  # Split date range into intervals of this many days
    articles_per_interval=20,  # Maximum number of articles to fetch per interval (e.g., per 7-day period)
)

gdelt_question_generator = QuestionGenerator(
    instructions=(
        "Generate forward-looking questions about global events and international news. "
        "Questions should focus on future outcomes that can be verified."
    ),
    examples=[
        "Will the conflict in region X escalate in the next month?",
        "Will country Y sign the trade agreement this quarter?",
        "Will the international summit achieve its stated goals?",
    ],
    bad_examples=[
        "What happened in the conflict?",
        "When was the trade agreement signed?",
        "Who attended the summit?",
    ],
    filter_=FilterCriteria(
        rubric="The question should be forward-looking and about future global events",
        min_score=0.7
    ),
    answer_type=answer_type,
)

gdelt_pipeline_config = QuestionPipeline(
    seed_generator=gdelt_seed_generator,
    question_generator=gdelt_question_generator,
    labeler=labeler,
    renderer=renderer,
)

## Run the Pipelines

You can run either pipeline configuration. Both work the same way - they just use different data sources.

In [15]:
search_dataset = lr.transforms.run(news_search_pipeline_config, max_questions=10, name="News search")

gdelt_dataset = lr.transforms.run(gdelt_pipeline_config, max_questions=10, name="GDELT")

Output()

Output()

> Note: This can take a few minutes to complete processing.

## View Results

Inspect the generated questions and answers. Each sample contains `seed`, `question`, `label`, `prompt`, and optional `context` and `meta` fields.

In [16]:
%pip install pandas

from IPython.display import clear_output
clear_output()

In [ ]:
import pandas as pd

news_samples = search_dataset.download()
news_rows = search_dataset.flattened()
news_df = pd.DataFrame(news_rows)

print(f"Generated {search_dataset.num_rows} samples (%.1f%% valid)\n" % (search_dataset.valid_count() / search_dataset.num_rows * 100))

cols = ["question_text", "answer", "label_confidence", "is_valid", "invalid_reason"]
news_df[[c for c in cols if c in news_df.columns]]

Generated 10 samples (70.0% valid)



,Question,Answer,Confidence,Valid,Invalid Reason
0,Will Amazon begin offering internet services t...,1,0.9,True,NaN
1,Will Alibaba Group Holding Ltd. announce a new...,1,1.0,False,Resolution date is before seed creation date
2,Will EBANX officially launch its AI-driven pay...,1,1.0,True,NaN
3,Will Spotify begin displaying AI disclosures i...,1,1.0,True,NaN
4,Will Meta officially announce the formation of...,1,1.0,True,NaN
5,Will G42 release a third edition of 'The Futur...,1,0.9,True,NaN
6,Will the NIST CHIPS Research and Development O...,1,1.0,False,Resolution date is before seed creation date
7,Will IONOS integrate at least one additional M...,1,1.0,True,NaN
8,Will the first gigawatt of NVIDIA systems for ...,1,0.9,True,NaN
9,Will Captain D’s complete the deployment of th...,Undetermined,0.8,False,Undetermined label


In [18]:
gdelt_samples = gdelt_dataset.download()
gdelt_rows = gdelt_dataset.flattened()
gdelt_df = pd.DataFrame(gdelt_rows)

print(f"Generated {gdelt_dataset.num_rows} samples (%.1f%% valid)\n" % (gdelt_dataset.valid_count() / gdelt_dataset.num_rows * 100))

cols = ["question_text", "answer", "label_confidence", "is_valid", "invalid_reason"]
gdelt_df[[c for c in cols if c in gdelt_df.columns]]

Generated 20 samples (40.0% valid)



,Question,Answer,Confidence,Valid,Invalid Reason
0,Will the Maharashtra government extend the dea...,NaN,NaN,False,Score 0.50 below min_score 0.7
1,Will OpenAI announce an official advertising b...,1,0.9,True,NaN
2,Will the total fertilizer subsidy expenditure ...,0,0.9,True,NaN
3,Will the U.S. Senate pass a bill to fund food ...,0,1.0,True,NaN
4,Will Supplemental Nutrition Assistance Program...,0,0.9,True,NaN
5,Will US Vice President JD Vance visit Kenya be...,0,1.0,True,NaN
6,Will the three-member investigation team const...,NaN,NaN,False,Score 0.10 below min_score 0.7
7,Will the French authorities announce the recov...,Undetermined,1.0,False,Undetermined label
8,Will the Bharatiya Janata Party (BJP) candidat...,Undetermined,1.0,False,Undetermined label
9,Will a formal peace agreement between Ukraine ...,Undetermined,0.9,False,Undetermined label


## When to Use Each Approach

**Use `NewsSeedGenerator` (Google News) when:**
- You need recent, curated news articles
- You want more control over search queries
- You're working with smaller, focused datasets
- You need faster iteration on specific topics
- You want to target specific keywords or themes

**Use `GdeltSeedGenerator` (Top Aggregated News) when:**
- You need access to a very large number of articles
- You're analyzing global or international events
- You need historical data
- You want broader coverage across many sources
- You're working with large-scale datasets